In [1]:

# NOTEBOOK 10 — CELL 1: QUICK ENVIRONMENT CHECK
from pathlib import Path
import torch
import pandas as pd
import shutil

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Mount Google Drive only if it is not already mounted.
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# Check the files created before Notebook 10.
required_files = {
    "Dataset ZIP": PROJECT / "datasets/archive.zip",
    "Original data split": PROJECT / "results/CASIA_Thousand_data_split.csv",
    "Augmented training manifest": PROJECT / "results/CASIA_Thousand_augmented_train_manifest.csv",
    "Baseline checkpoint": PROJECT / "models/CASIA_Thousand_baseline_resnet18.pth",
}

for name, path in required_files.items():
    print(f"{'FOUND' if path.is_file() else 'MISSING'} — {name}: {path}")

if (PROJECT / "results/CASIA_Thousand_augmented_train_manifest.csv").is_file():
    manifest = pd.read_csv(
        PROJECT / "results/CASIA_Thousand_augmented_train_manifest.csv"
    )
    print("\nManifest rows:", len(manifest))
    print("Sources:", manifest["source"].value_counts().to_dict())

print("\nColab disk free:",
      f"{shutil.disk_usage('/content').free / (1024**3):.2f} GB")
print("\nEnvironment check complete. No training started.")


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Mounted at /content/drive
FOUND — Dataset ZIP: /content/drive/MyDrive/Ocular_Project/datasets/archive.zip
FOUND — Original data split: /content/drive/MyDrive/Ocular_Project/results/CASIA_Thousand_data_split.csv
FOUND — Augmented training manifest: /content/drive/MyDrive/Ocular_Project/results/CASIA_Thousand_augmented_train_manifest.csv
FOUND — Baseline checkpoint: /content/drive/MyDrive/Ocular_Project/models/CASIA_Thousand_baseline_resnet18.pth

Manifest rows: 16864
Sources: {'real': 12000, 'diffusion_ddim50': 4800, 'diffusion_pilot': 64}

Colab disk free: 70.28 GB

Environment check complete. No training started.


In [2]:

# ============================================================
# NOTEBOOK 10 — CELL 2
# Prepare local dataset, cache synthetic images, build loaders
# ============================================================

import os, zipfile, shutil, json, time
from pathlib import Path
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
RESULTS = PROJECT / "results"
ZIP_PATH = PROJECT / "datasets/archive.zip"
SPLIT_CSV = RESULTS / "CASIA_Thousand_data_split.csv"
MANIFEST_CSV = RESULTS / "CASIA_Thousand_augmented_train_manifest.csv"

LOCAL_ROOT = Path("/content/ocular_project_local")
EXTRACT_DIR = LOCAL_ROOT / "dataset_from_zip"
LOCAL_SYNTHETIC = LOCAL_ROOT / "synthetic"
OUT_DIR = RESULTS / "recognition_with_augmented_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SYNTHETIC.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0  # reliable in Colab; avoids multiprocessing worker hangs

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------- 1. Extract dataset if needed --------------------
sample = next(EXTRACT_DIR.rglob("S5000L00.jpg"), None)
if sample is None:
    print("\nExtracting existing dataset ZIP to local disk...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    sample = next(EXTRACT_DIR.rglob("S5000L00.jpg"), None)

if sample is None:
    raise FileNotFoundError("Could not locate S5000L00.jpg after extraction.")

# Actual dataset root contains folders such as 000, 001, ...
LOCAL_DATASET = sample.parents[2]
print("Local dataset root:", LOCAL_DATASET)
print("Dataset sample verified:", sample.is_file())

# -------------------- 2. Load manifests --------------------
split_df = pd.read_csv(SPLIT_CSV)
manifest_df = pd.read_csv(MANIFEST_CSV)

split_df["identity"] = split_df["identity"].astype(str).str.strip()
split_df["split"] = split_df["split"].astype(str).str.lower().str.strip()
manifest_df["identity"] = manifest_df["identity"].astype(str).str.strip()
manifest_df["source"] = manifest_df["source"].astype(str).str.lower().str.strip()

DRIVE_DATASET = (
    PROJECT / "datasets" / "CASIA-Iris-Thousand"
    / "CASIA-Iris-Thousand" / "CASIA-Iris-Thousand"
)

def original_to_local(path):
    rel = Path(str(path)).relative_to(DRIVE_DATASET)
    return str(LOCAL_DATASET / rel)

split_df["local_path"] = split_df["image_path"].map(original_to_local)

# -------------------- 3. Cache synthetic images locally --------------------
synthetic_df = manifest_df[
    manifest_df["source"].isin(["diffusion_ddim50", "diffusion_pilot"])
].copy()

copied, missing = 0, 0
total = len(synthetic_df)
print(f"\nCaching {total} synthetic images locally...")

for i, src_text in enumerate(synthetic_df["image_path"].astype(str), start=1):
    src = Path(src_text)
    dest = LOCAL_SYNTHETIC / src.name

    if dest.exists():
        continue
    if src.is_file():
        shutil.copy2(src, dest)
        copied += 1
    else:
        missing += 1

    if i % 500 == 0:
        print(f"Checked {i}/{total} synthetic entries; copied {copied}")

print("New synthetic files copied:", copied)
print("Missing synthetic files:", missing)
if missing:
    raise FileNotFoundError(
        f"{missing} synthetic images missing from Drive. "
        "Check the Notebook 09 manifest."
    )

synthetic_df["local_path"] = synthetic_df["image_path"].map(
    lambda p: str(LOCAL_SYNTHETIC / Path(str(p)).name)
)

# -------------------- 4. Build class mapping and splits --------------------
classes = sorted(split_df["identity"].unique().tolist())
label_to_idx = {identity: i for i, identity in enumerate(classes)}
NUM_CLASSES = len(classes)

original_train = split_df[split_df["split"] == "train"].copy()
validation_df = split_df[split_df["split"] == "validation"].copy()
test_df = split_df[split_df["split"] == "test"].copy()

train_df = pd.concat([
    original_train[["local_path", "identity"]].rename(
        columns={"local_path": "path"}
    ),
    synthetic_df[synthetic_df["identity"].isin(label_to_idx)][
        ["local_path", "identity"]
    ].rename(columns={"local_path": "path"})
], ignore_index=True).drop_duplicates("path").reset_index(drop=True)

validation_df = validation_df[["local_path", "identity"]].rename(
    columns={"local_path": "path"}
).reset_index(drop=True)
test_df = test_df[["local_path", "identity"]].rename(
    columns={"local_path": "path"}
).reset_index(drop=True)

for df in [train_df, validation_df, test_df]:
    df["target"] = df["identity"].map(label_to_idx)

# -------------------- 5. Validate data separation --------------------
train_paths = set(train_df["path"])
val_paths = set(validation_df["path"])
test_paths = set(test_df["path"])

assert not train_paths.intersection(val_paths), "Train/validation leakage!"
assert not train_paths.intersection(test_paths), "Train/test leakage!"
assert not val_paths.intersection(test_paths), "Validation/test overlap!"

# Sample checks only, rather than repeatedly stat() every Drive file.
for name, df in [
    ("train", train_df), ("validation", validation_df), ("test", test_df)
]:
    sample_paths = df["path"].head(25)
    count = sum(os.path.isfile(p) for p in sample_paths)
    print(f"{name} local sample check: {count}/{len(sample_paths)}")
    if count != len(sample_paths):
        raise FileNotFoundError(f"Some {name} paths are missing locally.")

# -------------------- 6. Create datasets and loaders --------------------
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

class OcularDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame[["path", "target"]].reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["path"]) as im:
            x = transform(im.convert("L"))
        return x, int(row["target"])

train_loader = DataLoader(
    OcularDataset(train_df), batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    OcularDataset(validation_df), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    OcularDataset(test_df), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# -------------------- 7. Save audit and manifests --------------------
train_df.to_csv(OUT_DIR / "notebook10_training_manifest.csv", index=False)
validation_df.to_csv(OUT_DIR / "notebook10_validation_manifest.csv", index=False)
test_df.to_csv(OUT_DIR / "notebook10_test_manifest.csv", index=False)

audit = {
    "notebook": "10_Recognition_with_Augmented_Data",
    "classes": NUM_CLASSES,
    "original_train_images": len(original_train),
    "synthetic_images_included": len(synthetic_df),
    "combined_train_images": len(train_df),
    "validation_images": len(validation_df),
    "test_images": len(test_df),
    "synthetic_files_copied_this_run": copied,
    "missing_synthetic_files": missing,
    "train_validation_overlap": len(train_paths & val_paths),
    "train_test_overlap": len(train_paths & test_paths),
    "validation_test_overlap": len(val_paths & test_paths)
}
with open(OUT_DIR / "notebook10_dataset_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

# Verify one batch before any training.
xb, yb = next(iter(train_loader))

print("\n========== DATASET READY ==========")
print(json.dumps(audit, indent=2))
print("Batch image shape:", tuple(xb.shape))
print("Batch label shape:", tuple(yb.shape))
print("Batch label range:", int(yb.min()), "to", int(yb.max()))
print("Training loader batches:", len(train_loader))
print("Validation loader batches:", len(val_loader))
print("Test loader batches:", len(test_loader))


Device: cuda
GPU: Tesla T4

Extracting existing dataset ZIP to local disk...
Local dataset root: /content/ocular_project_local/dataset_from_zip/CASIA-Iris-Thousand/CASIA-Iris-Thousand
Dataset sample verified: True

Caching 4864 synthetic images locally...
Checked 500/4864 synthetic entries; copied 440
Checked 1000/4864 synthetic entries; copied 940
Checked 1500/4864 synthetic entries; copied 1440
Checked 2000/4864 synthetic entries; copied 1940
Checked 2500/4864 synthetic entries; copied 2440
Checked 3000/4864 synthetic entries; copied 2940
Checked 3500/4864 synthetic entries; copied 3440
Checked 4000/4864 synthetic entries; copied 3940
Checked 4500/4864 synthetic entries; copied 4440
New synthetic files copied: 4804
Missing synthetic files: 0
train local sample check: 25/25
validation local sample check: 25/25
test local sample check: 25/25

========== DATASET READY ==========
{
  "notebook": "10_Recognition_with_Augmented_Data",
  "classes": 2000,
  "original_train_images": 12000,
  

In [4]:

from pathlib import Path
import shutil
import pandas as pd
import json

# Reuse variables and paths from your previous cells
assert "manifest_df" in globals(), "Run the dataset preparation cell first."
assert "LOCAL_SYNTHETIC" in globals(), "Local synthetic cache path not found."

# Identify synthetic entries
synthetic_df = manifest_df[
    manifest_df["source"].isin(["diffusion_ddim50", "diffusion_pilot"])
].copy()

pilot_df = synthetic_df[synthetic_df["source"] == "diffusion_pilot"]
ddim_df = synthetic_df[synthetic_df["source"] == "diffusion_ddim50"]

print("Pilot images to fix:", len(pilot_df))
print("DDIM images to reuse:", len(ddim_df))

# DDIM filenames should already be unique.
ddim_names = ddim_df["image_path"].map(lambda p: Path(p).name)
if ddim_names.duplicated().any():
    raise RuntimeError(
        "DDIM filenames also collide. Stop and inspect their paths before training."
    )

# Rebuild the synthetic local paths safely.
def get_safe_local_path(row):
    src = Path(row["image_path"])
    if row["source"] == "diffusion_pilot":
        # Include identity folder to prevent sample_XX.png collisions.
        filename = f"pilot_{src.parent.name}_{src.name}"
    else:
        # Preserve existing cached DDIM filenames.
        filename = src.name
    return LOCAL_SYNTHETIC / filename

synthetic_df["local_path"] = synthetic_df.apply(get_safe_local_path, axis=1)

# Copy only pilot images to their new unique paths.
copied = 0
missing_sources = []

for _, row in pilot_df.iterrows():
    src = Path(row["image_path"])
    dest = get_safe_local_path(row)

    if not dest.is_file():
        if src.is_file():
            shutil.copy2(src, dest)
            copied += 1
        else:
            missing_sources.append(str(src))

    if copied and copied % 16 == 0:
        print(f"Copied {copied}/{len(pilot_df)} pilot images...")

# Validate all synthetic mappings.
missing_local = [
    str(p) for p in synthetic_df["local_path"]
    if not Path(p).is_file()
]
duplicate_local = synthetic_df["local_path"].astype(str).duplicated().sum()

print("\n========== CACHE CORRECTION AUDIT ==========")
print("Pilot files newly copied:", copied)
print("Missing source pilot files:", len(missing_sources))
print("Missing local synthetic files:", len(missing_local))
print("Duplicate local paths:", duplicate_local)
print("Unique synthetic local paths:", synthetic_df["local_path"].nunique())

if missing_sources:
    print("Example missing source:", missing_sources[:3])

assert not missing_sources, "Some original pilot images are missing."
assert not missing_local, "Some synthetic cache files are missing."
assert duplicate_local == 0, "Synthetic local paths still collide."

# Rebuild training manifest without collapsing distinct images.
real_df = manifest_df[manifest_df["source"] == "real"].copy()
real_df["local_path"] = real_df["image_path"].map(
    lambda p: Path(p)  # Original-image local paths are set in the loader.
)

train_df = pd.concat(
    [real_df, synthetic_df],
    ignore_index=True
)

print("\nExpected combined training rows:", len(manifest_df))
print("Actual combined training rows:", len(train_df))
print("Real training rows:", len(real_df))
print("Synthetic training rows:", len(synthetic_df))

assert len(train_df) == len(manifest_df) == 16864, (
    "Training row count mismatch; stop before training."
)

print("\nBro, cache mapping is corrected. Training has NOT started yet.")


Pilot images to fix: 64
DDIM images to reuse: 4800
Copied 16/64 pilot images...
Copied 32/64 pilot images...
Copied 48/64 pilot images...
Copied 64/64 pilot images...

========== CACHE CORRECTION AUDIT ==========
Pilot files newly copied: 64
Missing source pilot files: 0
Missing local synthetic files: 0
Duplicate local paths: 0
Unique synthetic local paths: 4864

Expected combined training rows: 16864
Actual combined training rows: 16864
Real training rows: 12000
Synthetic training rows: 4864

Bro, cache mapping is corrected. Training has NOT started yet.


In [8]:

# NOTEBOOK 10 — CELL 2B: FINAL DATASET PATH + LOADER FIX

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch
import numpy as np

# Exact dataset root confirmed by your Colab screenshot.
LOCAL_ROOT = Path(
    "/content/ocular_project_local/dataset_from_zip/"
    "CASIA-Iris-Thousand/CASIA-Iris-Thousand"
)

SPLIT_FILE = Path(
    "/content/drive/MyDrive/Ocular_Project/results/"
    "CASIA_Thousand_data_split.csv"
)

# ---------- 1. Verify required existing variables ----------
for name in ["train_df", "synthetic_df", "label_to_idx"]:
    if name not in globals():
        raise RuntimeError(
            f"{name} is missing. Do not rerun extraction; "
            "check the earlier dataset/cache cells."
        )

sample = LOCAL_ROOT / "000" / "L" / "S5000L00.jpg"
print("Dataset root exists:", LOCAL_ROOT.is_dir())
print("Known sample exists:", sample.is_file())

if not sample.is_file():
    raise FileNotFoundError(f"Expected sample not found: {sample}")

# ---------- 2. Build local paths without scanning the disk repeatedly ----------
def original_local_path(row):
    source_path = Path(str(row["image_path"]))
    identity = str(row["identity"])
    subject, eye = identity.split("_", 1)

    candidate = LOCAL_ROOT / subject / eye / source_path.name
    if not candidate.is_file():
        raise FileNotFoundError(f"Original image missing: {candidate}")
    return str(candidate)

# ---------- 3. Rebuild training rows ----------
real_df = train_df[train_df["source"].astype(str) == "real"].copy()
real_df["local_path"] = [
    original_local_path(row) for _, row in real_df.iterrows()
]

syn_df = synthetic_df.copy()
syn_df["local_path"] = syn_df["local_path"].astype(str)

train_fixed = pd.concat([real_df, syn_df], ignore_index=True)

# ---------- 4. Rebuild validation/test rows ----------
split_df = pd.read_csv(SPLIT_FILE)

val_df = split_df[split_df["split"] == "validation"].copy()
test_df_fixed = split_df[split_df["split"] == "test"].copy()

val_df["local_path"] = [
    original_local_path(row) for _, row in val_df.iterrows()
]
test_df_fixed["local_path"] = [
    original_local_path(row) for _, row in test_df_fixed.iterrows()
]

# ---------- 5. Validate counts, paths, and identity labels ----------
class_to_idx = {str(k): int(v) for k, v in label_to_idx.items()}
NUM_CLASSES = len(class_to_idx)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

expected_counts = {
    "train": (train_fixed, 16864),
    "validation": (val_df, 4000),
    "test": (test_df_fixed, 4000),
}

for name, (frame, expected) in expected_counts.items():
    missing = sum(
        not Path(p).is_file() for p in frame["local_path"]
    )
    unknown = (
        set(frame["identity"].astype(str)) - set(class_to_idx.keys())
    )

    print(
        f"{name}: rows={len(frame)}, missing_paths={missing}, "
        f"unknown_identities={len(unknown)}"
    )

    if len(frame) != expected:
        raise RuntimeError(
            f"{name} count is {len(frame)}, expected {expected}."
        )
    if missing:
        raise RuntimeError(f"{name} contains missing image paths.")
    if unknown:
        raise RuntimeError(
            f"{name} contains identities absent from label_to_idx: "
            f"{list(unknown)[:5]}"
        )

if syn_df["local_path"].duplicated().any():
    raise RuntimeError("Duplicate synthetic local paths detected.")

# ---------- 6. Dataset and loaders ----------
class OcularLocalDataset(Dataset):
    def __init__(self, frame, mapping):
        self.frame = frame.reset_index(drop=True)
        self.mapping = mapping

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        with Image.open(row["local_path"]) as im:
            im = im.convert("L").resize((224, 224))
            arr = np.asarray(im, dtype=np.float32) / 255.0

        image = torch.from_numpy(arr).unsqueeze(0)
        label = self.mapping[str(row["identity"])]

        return image, torch.tensor(label, dtype=torch.long)

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    OcularLocalDataset(train_fixed, class_to_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    OcularLocalDataset(val_df, class_to_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    OcularLocalDataset(test_df_fixed, class_to_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# ---------- 7. Final smoke test ----------
images, labels = next(iter(train_loader))

print("\n========== FINAL LOADER AUDIT ==========")
print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))
print("Synthetic samples:", len(syn_df))
print("Classes:", NUM_CLASSES)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("Batch image shape:", tuple(images.shape))
print("Batch label shape:", tuple(labels.shape))
print("\nLoader setup complete. Training has NOT started.")


Dataset root exists: True
Known sample exists: True
train: rows=16864, missing_paths=0, unknown_identities=0
validation: rows=4000, missing_paths=0, unknown_identities=0
test: rows=4000, missing_paths=0, unknown_identities=0

========== FINAL LOADER AUDIT ==========
Train samples: 16864
Validation samples: 4000
Test samples: 4000
Synthetic samples: 4864
Classes: 2000
Train batches: 527
Validation batches: 125
Test batches: 125
Batch image shape: (32, 1, 224, 224)
Batch label shape: (32,)

Loader setup complete. Training has NOT started.


In [9]:

# NOTEBOOK 10 — CELL 3: VERIFY CACHE + TRAIN AUGMENTED RESNET18

import os, json, time, shutil
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torchvision import models

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
RESULTS = PROJECT / "results"
MODELS = PROJECT / "models"

OUT_DIR = RESULTS / "recognition_with_augmented_data"
MODEL_DIR = MODELS / "recognition_with_augmented_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = RESULTS / "CASIA_Thousand_augmented_train_manifest.csv"
BASELINE_CHECKPOINT = MODELS / "CASIA_Thousand_baseline_resnet18.pth"

CHECKPOINT = MODEL_DIR / "CASIA_Thousand_augmented_resnet18_latest.pth"
BEST_CHECKPOINT = MODEL_DIR / "CASIA_Thousand_augmented_resnet18_best.pth"
HISTORY_FILE = OUT_DIR / "notebook10_training_history.csv"
AUDIT_FILE = OUT_DIR / "notebook10_dataset_audit.json"

# ---------------------------------------------------------
# 1. Verify the corrected synthetic cache from Cell 2 + fix
# ---------------------------------------------------------
if "synthetic_df" not in globals():
    raise RuntimeError(
        "Corrected synthetic_df not found. Run the cache-correction cell first."
    )

if "train_df" not in globals():
    raise RuntimeError(
        "Corrected train_df not found. Run the cache-correction cell first."
    )

manifest = pd.read_csv(MANIFEST_FILE)
expected_synthetic = manifest[
    manifest["source"].astype(str).str.lower().isin(
        ["diffusion_ddim50", "diffusion_pilot"]
    )
]

if len(synthetic_df) != len(expected_synthetic):
    raise RuntimeError("Synthetic row count differs from the saved manifest.")

if synthetic_df["local_path"].astype(str).duplicated().any():
    raise RuntimeError("Duplicate synthetic local paths detected.")

missing = [
    str(p) for p in synthetic_df["local_path"]
    if not Path(p).is_file()
]
if missing:
    raise RuntimeError(
        f"{len(missing)} synthetic images are missing. Example: {missing[:3]}"
    )

if len(train_df) != len(manifest):
    raise RuntimeError(
        f"Training row mismatch: {len(train_df)} vs {len(manifest)}"
    )

print("Synthetic images:", len(synthetic_df))
print("Unique synthetic paths:", synthetic_df["local_path"].nunique())
print("Missing synthetic files:", len(missing))
print("Combined training rows:", len(train_df))
print("Cache verification passed.")

# ---------------------------------------------------------
# 2. Check loaders and label mapping
# ---------------------------------------------------------
for name in ["train_loader", "val_loader", "test_loader",
             "NUM_CLASSES", "DEVICE", "label_to_idx"]:
    if name not in globals():
        raise RuntimeError(
            f"{name} is missing. Run the dataset preparation cell first."
        )

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))
print("Classes:", NUM_CLASSES)
print("Device:", DEVICE)

# Confirm the loader reflects the corrected combined dataset.
if len(train_loader.dataset) != 16864:
    raise RuntimeError(
        "The current train_loader does not contain 16,864 samples. "
        "Rebuild the loaders from the corrected train_df before training."
    )

# ---------------------------------------------------------
# 3. Build model
# ---------------------------------------------------------
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# Inspect baseline checkpoint compatibility.
# We reuse it only if its architecture and label mapping can be verified.
use_baseline = False

if BASELINE_CHECKPOINT.is_file():
    try:
        baseline = torch.load(
            BASELINE_CHECKPOINT, map_location="cpu", weights_only=False
        )

        if isinstance(baseline, dict) and "model_state_dict" in baseline:
            baseline_state = baseline["model_state_dict"]
        elif isinstance(baseline, dict) and "state_dict" in baseline:
            baseline_state = baseline["state_dict"]
        elif isinstance(baseline, dict):
            baseline_state = baseline
        else:
            baseline_state = None

        if baseline_state is not None:
            # Strict loading ensures the final layer and all shapes match.
            model.load_state_dict(baseline_state, strict=True)
            use_baseline = True
            print("Compatible baseline weights loaded.")
        else:
            print("Baseline format unrecognized; using fresh initialization.")

    except Exception as e:
        print("Baseline checkpoint not compatible:", str(e)[:250])
        print("Using fresh initialization instead.")
else:
    print("Baseline checkpoint not found; using fresh initialization.")

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4 if use_baseline else 1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

# If a prior run saved a checkpoint, resume it when compatible.
start_epoch = 1
best_val_acc = -1.0
history = []

if CHECKPOINT.is_file():
    try:
        saved = torch.load(
            CHECKPOINT, map_location=DEVICE, weights_only=False
        )
        if (saved.get("num_classes") == NUM_CLASSES
                and "model_state_dict" in saved):
            model.load_state_dict(saved["model_state_dict"], strict=True)
            if "optimizer_state_dict" in saved:
                optimizer.load_state_dict(saved["optimizer_state_dict"])
            start_epoch = int(saved.get("epoch", 0)) + 1
            best_val_acc = float(saved.get("best_val_accuracy", -1.0))
            if HISTORY_FILE.is_file():
                history = pd.read_csv(HISTORY_FILE).to_dict("records")
            print(f"Resuming saved training at epoch {start_epoch}.")
    except Exception as e:
        print("No compatible training checkpoint to resume:", str(e)[:200])

# ---------------------------------------------------------
# 4. Train, validate, and save every epoch
# ---------------------------------------------------------
EPOCHS = 10
start_all = time.time()

if start_epoch > EPOCHS:
    print("Training already reached the configured epoch count.")
else:
    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for step, (images, labels) in enumerate(train_loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * labels.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

            if step % 50 == 0 or step == len(train_loader):
                print(
                    f"\rEpoch {epoch}/{EPOCHS} | "
                    f"train batch {step}/{len(train_loader)}",
                    end="", flush=True
                )

        train_loss = train_loss_sum / max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)

        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for step, (images, labels) in enumerate(val_loader, start=1):
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * labels.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_sum / max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)
        scheduler.step(val_loss)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "validation_loss": val_loss,
            "validation_accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.time() - epoch_start
        }
        history = [r for r in history if int(r["epoch"]) != epoch]
        history.append(row)
        pd.DataFrame(history).sort_values("epoch").to_csv(
            HISTORY_FILE, index=False
        )

        state = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "num_classes": NUM_CLASSES,
            "label_to_idx": class_to_idx,
            "validation_accuracy": val_acc,
            "validation_loss": val_loss,
            "best_val_accuracy": best_val_acc,
            "initialized_from_baseline": use_baseline
        }

        # Save the latest checkpoint after each epoch.
        torch.save(state, CHECKPOINT)

        if improved or not BEST_CHECKPOINT.is_file():
            torch.save(state, BEST_CHECKPOINT)

        # Record a lightweight audit as well.
        audit = {
            "training_samples": len(train_loader.dataset),
            "validation_samples": len(val_loader.dataset),
            "test_samples": len(test_loader.dataset),
            "synthetic_samples": len(synthetic_df),
            "num_classes": NUM_CLASSES,
            "initialized_from_baseline": use_baseline,
            "completed_epoch": epoch,
            "best_validation_accuracy": best_val_acc
        }
        AUDIT_FILE.write_text(json.dumps(audit, indent=2))

        print(
            f"\nEpoch {epoch}/{EPOCHS} | "
            f"train loss={train_loss:.4f}, acc={train_acc*100:.2f}% | "
            f"val loss={val_loss:.4f}, acc={val_acc*100:.2f}% | "
            f"time={(time.time()-epoch_start)/60:.1f} min"
        )
        print("Checkpoint saved to Drive.")

print("\nTraining cell finished.")
print("Latest checkpoint:", CHECKPOINT)
print("Best checkpoint:", BEST_CHECKPOINT)
print("History:", HISTORY_FILE)
print("Dataset audit:", AUDIT_FILE)


Synthetic images: 4864
Unique synthetic paths: 4864
Missing synthetic files: 0
Combined training rows: 16864
Cache verification passed.
Train samples: 16864
Validation samples: 4000
Test samples: 4000
Classes: 2000
Device: cuda
Compatible baseline weights loaded.
Epoch 1/10 | train batch 527/527
Epoch 1/10 | train loss=4.3607, acc=50.30% | val loss=3.0168, acc=50.58% | time=1.3 min
Checkpoint saved to Drive.
Epoch 2/10 | train batch 527/527
Epoch 2/10 | train loss=2.8459, acc=63.51% | val loss=2.5452, acc=56.75% | time=1.2 min
Checkpoint saved to Drive.
Epoch 3/10 | train batch 527/527
Epoch 3/10 | train loss=2.5475, acc=65.99% | val loss=2.3003, acc=60.45% | time=1.3 min
Checkpoint saved to Drive.
Epoch 4/10 | train batch 527/527
Epoch 4/10 | train loss=2.3981, acc=67.68% | val loss=2.2779, acc=60.77% | time=1.3 min
Checkpoint saved to Drive.
Epoch 5/10 | train batch 527/527
Epoch 5/10 | train loss=2.3023, acc=68.69% | val loss=2.1312, acc=62.82% | time=1.3 min
Checkpoint saved to Dri

In [10]:

# NOTEBOOK 10 — CELL 4: EVALUATE BEST AUGMENTED MODEL ON TEST SET

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
MODEL_DIR = PROJECT / "models/recognition_with_augmented_data"
OUT_DIR = PROJECT / "results/recognition_with_augmented_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = (
    MODEL_DIR / "CASIA_Thousand_augmented_resnet18_best.pth"
)

RESULTS_JSON = OUT_DIR / "notebook10_test_metrics.json"
PREDICTIONS_CSV = OUT_DIR / "notebook10_test_predictions.csv"
CLASS_METRICS_CSV = OUT_DIR / "notebook10_per_class_metrics.csv"
CONFUSION_NPY = OUT_DIR / "notebook10_confusion_matrix.npy"

assert BEST_CHECKPOINT.is_file(), (
    f"Best checkpoint not found: {BEST_CHECKPOINT}"
)
assert "test_loader" in globals(), (
    "test_loader is missing. Run the corrected loader cell first."
)
assert "class_to_idx" in globals(), (
    "class_to_idx is missing. Run the corrected loader cell first."
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(class_to_idx)

print("Device:", DEVICE)
print("Test samples:", len(test_loader.dataset))
print("Classes:", NUM_CLASSES)
print("Loading best checkpoint...")

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

# Verify checkpoint metadata before evaluating.
if checkpoint.get("num_classes") != NUM_CLASSES:
    raise RuntimeError(
        f"Checkpoint classes ({checkpoint.get('num_classes')}) "
        f"do not match current mapping ({NUM_CLASSES})."
    )

saved_mapping = checkpoint.get("label_to_idx")
if saved_mapping is not None:
    saved_mapping = {str(k): int(v) for k, v in saved_mapping.items()}
    current_mapping = {str(k): int(v) for k, v in class_to_idx.items()}

    if saved_mapping != current_mapping:
        raise RuntimeError(
            "Checkpoint label mapping differs from the current mapping. "
            "Do not evaluate until the mappings are reconciled."
        )

model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model = model.to(DEVICE)
model.eval()

# Evaluate once on the held-out test set.
all_labels = []
all_predictions = []
all_probabilities = []
start = time.time()

with torch.inference_mode():
    for step, (images, labels) in enumerate(test_loader, start=1):
        images = images.to(DEVICE, non_blocking=True)
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy().tolist())
        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_probabilities.extend(
            probabilities.max(dim=1).values.cpu().numpy().tolist()
        )

        if step % 25 == 0 or step == len(test_loader):
            print(f"Evaluated test batches: {step}/{len(test_loader)}")

y_true = np.asarray(all_labels)
y_pred = np.asarray(all_predictions)
confidence = np.asarray(all_probabilities)

accuracy = accuracy_score(y_true, y_pred)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        average="macro",
        zero_division=0
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
)

cm = confusion_matrix(
    y_true, y_pred, labels=np.arange(NUM_CLASSES)
)
np.save(CONFUSION_NPY, cm)

# Save human-readable predictions.
idx_to_class = {v: k for k, v in class_to_idx.items()}
predictions_df = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": y_pred,
    "true_identity": [idx_to_class[int(i)] for i in y_true],
    "predicted_identity": [idx_to_class[int(i)] for i in y_pred],
    "correct": y_true == y_pred,
    "predicted_class_confidence": confidence
})
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# Per-class metrics.
per_precision, per_recall, per_f1, per_support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0
    )
)

class_metrics_df = pd.DataFrame({
    "class_index": np.arange(NUM_CLASSES),
    "identity": [idx_to_class[i] for i in range(NUM_CLASSES)],
    "precision": per_precision,
    "recall": per_recall,
    "f1_score": per_f1,
    "support": per_support
})
class_metrics_df.to_csv(CLASS_METRICS_CSV, index=False)

metrics = {
    "model": "augmented_resnet18_best",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "checkpoint_validation_accuracy": float(
        checkpoint["validation_accuracy"]
    ),
    "test_samples": int(len(y_true)),
    "num_classes": int(NUM_CLASSES),
    "accuracy": float(accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "weighted_precision": float(weighted_precision),
    "weighted_recall": float(weighted_recall),
    "weighted_f1": float(weighted_f1),
    "mean_predicted_class_confidence": float(confidence.mean()),
    "evaluation_seconds": round(time.time() - start, 2),
    "confusion_matrix_shape": list(cm.shape),
    "initialized_from_baseline": checkpoint.get(
        "initialized_from_baseline", None
    )
}

RESULTS_JSON.write_text(json.dumps(metrics, indent=2))

print("\n========== AUGMENTED MODEL TEST RESULTS ==========")
print(f"Checkpoint epoch:       {checkpoint['epoch']}")
print(f"Test accuracy:          {accuracy*100:.2f}%")
print(f"Macro precision:        {macro_precision*100:.2f}%")
print(f"Macro recall:           {macro_recall*100:.2f}%")
print(f"Macro F1-score:         {macro_f1*100:.2f}%")
print(f"Weighted precision:     {weighted_precision*100:.2f}%")
print(f"Weighted recall:        {weighted_recall*100:.2f}%")
print(f"Weighted F1-score:      {weighted_f1*100:.2f}%")
print(f"Evaluation time:        {metrics['evaluation_seconds']} sec")

print("\nSaved files:")
print("Metrics:", RESULTS_JSON)
print("Predictions:", PREDICTIONS_CSV)
print("Per-class metrics:", CLASS_METRICS_CSV)
print("Confusion matrix:", CONFUSION_NPY)


Device: cuda
Test samples: 4000
Classes: 2000
Loading best checkpoint...
Evaluated test batches: 25/125
Evaluated test batches: 50/125
Evaluated test batches: 75/125
Evaluated test batches: 100/125
Evaluated test batches: 125/125

========== AUGMENTED MODEL TEST RESULTS ==========
Checkpoint epoch:       10
Test accuracy:          55.83%
Macro precision:        50.45%
Macro recall:           55.83%
Macro F1-score:         50.38%
Weighted precision:     50.45%
Weighted recall:        55.83%
Weighted F1-score:      50.38%
Evaluation time:        19.97 sec

Saved files:
Metrics: /content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/notebook10_test_metrics.json
Predictions: /content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/notebook10_test_predictions.csv
Per-class metrics: /content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/notebook10_per_class_metrics.csv
Confusion matrix: /content/drive/MyDrive/Ocular_Project/r

In [11]:

# NOTEBOOK 10 — CELL 5: BASELINE VS AUGMENTED MODEL COMPARISON

import json
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
BASELINE_PATH = PROJECT / "models/CASIA_Thousand_baseline_resnet18.pth"
AUGMENTED_PATH = (
    PROJECT / "models/recognition_with_augmented_data/"
    "CASIA_Thousand_augmented_resnet18_best.pth"
)
OUT_DIR = PROJECT / "results/recognition_with_augmented_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert BASELINE_PATH.is_file(), f"Missing baseline: {BASELINE_PATH}"
assert AUGMENTED_PATH.is_file(), f"Missing augmented model: {AUGMENTED_PATH}"
assert "test_loader" in globals() and "class_to_idx" in globals()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = len(class_to_idx)

def load_checkpoint(path):
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "model_state_dict" in obj:
        state = obj["model_state_dict"]
        mapping = obj.get("label_to_idx")
    elif isinstance(obj, dict) and "state_dict" in obj:
        state = obj["state_dict"]
        mapping = obj.get("label_to_idx")
    else:
        state = obj
        mapping = None
    return obj, state, mapping

def evaluate_checkpoint(path, model_name):
    metadata, state, saved_mapping = load_checkpoint(path)

    # If checkpoint contains its label mapping, require exact agreement.
    if saved_mapping is not None:
        saved_mapping = {str(k): int(v) for k, v in saved_mapping.items()}
        current_mapping = {str(k): int(v) for k, v in class_to_idx.items()}
        if saved_mapping != current_mapping:
            raise RuntimeError(
                f"{model_name}: saved label mapping differs from current mapping. "
                "Do not compare until mappings are reconciled."
            )
    else:
        print(
            f"WARNING: {model_name} checkpoint has no saved label mapping. "
            "Its class-index semantics cannot be verified from metadata alone."
        )

    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    model.load_state_dict(state, strict=True)
    model = model.to(DEVICE).eval()

    true_labels, predictions = [], []

    with torch.inference_mode():
        for images, labels in test_loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            true_labels.extend(labels.cpu().numpy().tolist())
            predictions.extend(preds.cpu().numpy().tolist())

    accuracy = accuracy_score(true_labels, predictions)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        true_labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0
    )

    return {
        "model": model_name,
        "test_samples": len(true_labels),
        "accuracy": float(accuracy),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "label_mapping_verified": saved_mapping is not None
    }

print("Evaluating original baseline...")
baseline_results = evaluate_checkpoint(
    BASELINE_PATH, "Original baseline"
)

print("\nEvaluating best augmented model...")
augmented_results = evaluate_checkpoint(
    AUGMENTED_PATH, "Augmented fine-tuned model"
)

comparison = pd.DataFrame([baseline_results, augmented_results])
comparison.to_csv(OUT_DIR / "baseline_vs_augmented_test_metrics.csv", index=False)

print("\n========== TEST SET COMPARISON ==========")
display(comparison)

if (baseline_results["label_mapping_verified"]
        and augmented_results["label_mapping_verified"]):
    delta = (
        augmented_results["accuracy"] - baseline_results["accuracy"]
    ) * 100
    print(f"\nAccuracy change: {delta:+.2f} percentage points")
    print("Comparison saved to:", OUT_DIR / "baseline_vs_augmented_test_metrics.csv")
else:
    print(
        "\nComparison saved, but verify class-index mappings before "
        "drawing conclusions about improvement."
    )


Evaluating original baseline...

Evaluating best augmented model...

========== TEST SET COMPARISON ==========


,model,test_samples,accuracy,macro_precision,macro_recall,macro_f1,label_mapping_verified
0,Original baseline,4000,0.03075,0.018456,0.03075,0.017781,False
1,Augmented fine-tuned model,4000,0.55825,0.504488,0.55825,0.503830,True



Comparison saved, but verify class-index mappings before drawing conclusions about improvement.


In [12]:

# NOTEBOOK 10 — CELL 6: INSPECT BASELINE LABEL MAPPING

from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
baseline_path = PROJECT / "models/CASIA_Thousand_baseline_resnet18.pth"

checkpoint = torch.load(
    baseline_path, map_location="cpu", weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level checkpoint keys:")
    print(list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")
            print("Example keys:", list(value.keys())[:8])
        elif isinstance(value, (str, int, float, bool, list, tuple)):
            print(f"{key}: {str(value)[:300]}")

print("\nCurrent mapping examples:")
print(list(class_to_idx.items())[:12])

print("\nCurrent mapping size:", len(class_to_idx))

# Look for saved label mappings or class-name lists.
if isinstance(checkpoint, dict):
    for key in checkpoint:
        value = checkpoint[key]
        if any(term in key.lower() for term in [
            "label", "class", "identity", "mapping", "idx"
        ]):
            print(f"\nPotential mapping field: {key}")
            print(str(value)[:1000])

print("\nDiagnostic complete. No model was trained or modified.")


Checkpoint type: OrderedDict

Top-level checkpoint keys:
['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.1.conv1.weight', 'layer1.1.bn1.weight', 'layer1.1.bn1.bias', 'layer1.1.bn1.running_mean', 'layer1.1.bn1.running_var', 'layer1.1.bn1.num_batches_tracked', 'layer1.1.conv2.weight', 'layer1.1.bn2.weight', 'layer1.1.bn2.bias', 'layer1.1.bn2.running_mean', 'layer1.1.bn2.running_var', 'layer1.1.bn2.num_batches_tracked', 'layer2.0.conv1.weight', 'layer2.0.bn1.weight', 'layer2.0.bn1.bias', 'layer2.0.bn1.running_mean', 'layer2.0.bn1.running_var', 'layer2.0.bn1.num_batches_tracked', 'layer2.0.conv2.weight',

In [13]:

# NOTEBOOK 10 — CELL 7: SEARCH FOR BASELINE LABEL MAPPING

from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")

print("Searching project files for baseline mapping artifacts...\n")

# Search likely project locations, excluding the large image dataset.
search_roots = [
    PROJECT / "notebooks",
    PROJECT / "results",
    PROJECT / "models"
]

keywords = ("baseline", "label", "class", "mapping", "recognition")

for root in search_roots:
    if not root.exists():
        print("Folder not found:", root)
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".json", ".csv", ".txt", ".py", ".ipynb", ".pkl"
        }:
            continue

        if any(word in path.name.lower() for word in keywords):
            print(path)

print("\nSearching CSV headers for possible identity mappings...")
for path in (PROJECT / "results").rglob("*.csv"):
    try:
        header = pd.read_csv(path, nrows=0).columns.tolist()
        if any("label" in c.lower() or "identity" in c.lower()
               or "class" in c.lower() for c in header):
            print(path, "->", header)
    except Exception:
        pass

print("\nSearch complete. No files were modified.")


Searching project files for baseline mapping artifacts...

Folder not found: /content/drive/MyDrive/Ocular_Project/notebooks
/content/drive/MyDrive/Ocular_Project/results/CASIA_Thousand_baseline_training_history.json
/content/drive/MyDrive/Ocular_Project/results/CASIA_Thousand_baseline_test_results.csv
/content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/notebook10_per_class_metrics.csv
/content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/baseline_vs_augmented_test_metrics.csv

Searching CSV headers for possible identity mappings...
/content/drive/MyDrive/Ocular_Project/results/CASIA_Thousand_data_split.csv -> ['image_path', 'identity', 'subject', 'eye', 'split']
/content/drive/MyDrive/Ocular_Project/results/generation_records.csv -> ['identity_index', 'sample_index', 'seed', 'finite', 'mean', 'std', 'min', 'max', 'tensor_path', 'png_path']
/content/drive/MyDrive/Ocular_Project/results/identity_consistency_metrics.csv -> ['same_identity

In [14]:

# NOTEBOOK 10 — CELL 8: INSPECT SAVED BASELINE RESULTS

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
RESULTS = PROJECT / "results"

files_to_check = {
    "Baseline test results": RESULTS / "CASIA_Thousand_baseline_test_results.csv",
    "Baseline training history": RESULTS / "CASIA_Thousand_baseline_training_history.json",
}

for name, path in files_to_check.items():
    print(f"\n{'=' * 15} {name} {'=' * 15}")
    print("Path:", path)
    print("Exists:", path.is_file())

    if not path.is_file():
        continue

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        display(df.head(10))
    elif path.suffix.lower() == ".json":
        with open(path, "r") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2)[:8000])

print("\nInspection complete. No files modified.")



=============== Baseline test results ===============
Path: /content/drive/MyDrive/Ocular_Project/results/CASIA_Thousand_baseline_test_results.csv
Exists: True
Shape: (1, 11)


,Dataset,Experiment,Model,Training_Images,Validation_Images,Test_Images,Number_of_Identities,Test_Accuracy,Test_Precision,Test_Recall,Test_F1
0,CASIA-Iris-Thousand,Baseline,ResNet18,12000,4000,4000,2000,0.30175,0.272411,0.30175,0.260661



=============== Baseline training history ===============
Path: /content/drive/MyDrive/Ocular_Project/results/CASIA_Thousand_baseline_training_history.json
Exists: True
{
  "epochs": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10
  ],
  "train_loss": [
    7.653236549377441,
    7.427878885904948,
    7.013792049407959,
    6.089247694651286,
    5.173275248209635,
    4.179376382827758,
    2.8778432165781656,
    1.678010574022929,
    0.907708213965098,
    0.46241778691609703
  ],
  "train_accuracy": [
    0.00016666666666666666,
    0.0005833333333333334,
    0.0025,
    0.014583333333333334,
    0.05225,
    0.15483333333333332,
    0.36783333333333335,
    0.6068333333333333,
    0.783,
    0.8876666666666667
  ],
  "validation_loss": [
    7.532726543426514,
    7.38889175415039,
    6.821076419830322,
    6.445069190979004,
    6.4405277099609375,
    5.2955202255249025,
    4.967984575271607,
    7.99501123046875,
    3.5845281534194946,
    3.794358

In [15]:

# NOTEBOOK 10 — CELL 9: BASELINE VALIDATION SANITY CHECK

import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
BASELINE_CHECKPOINT = PROJECT / "models/CASIA_Thousand_baseline_resnet18.pth"

assert "val_loader" in globals(), "Run the dataset/loader setup first."
assert "DEVICE" in globals(), "DEVICE is missing."
assert "NUM_CLASSES" in globals(), "NUM_CLASSES is missing."

# Recreate the original grayscale ResNet18 architecture
baseline_model = models.resnet18(weights=None)
baseline_model.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
baseline_model.fc = nn.Linear(baseline_model.fc.in_features, NUM_CLASSES)

state = torch.load(
    BASELINE_CHECKPOINT, map_location="cpu", weights_only=False
)

if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]
elif isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

baseline_model.load_state_dict(state, strict=True)
baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()

correct, total = 0, 0

with torch.no_grad():
    for step, (images, labels) in enumerate(val_loader, start=1):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        predictions = baseline_model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        if step % 25 == 0 or step == len(val_loader):
            print(f"Validation batch {step}/{len(val_loader)}")

print("\n========== BASELINE VALIDATION CHECK ==========")
print("Samples:", total)
print(f"Accuracy with current label mapping: {100 * correct / total:.2f}%")
print("Reference validation accuracy from saved history: 42.175%")
print("No training performed; no checkpoints modified.")


Validation batch 25/125
Validation batch 50/125
Validation batch 75/125
Validation batch 100/125
Validation batch 125/125

========== BASELINE VALIDATION CHECK ==========
Samples: 4000
Accuracy with current label mapping: 3.38%
Reference validation accuracy from saved history: 42.175%
No training performed; no checkpoints modified.


In [16]:

# NOTEBOOK 10 — CELL 10: DIAGNOSE BASELINE PREPROCESSING
import torch
import torch.nn.functional as F

assert "baseline_model" in globals(), "Run the baseline validation sanity-check cell first."
assert "val_loader" in globals(), "Validation loader not found."

baseline_model.eval()
device = next(baseline_model.parameters()).device

# Inspect one batch using the exact current validation loader.
images, labels = next(iter(val_loader))
images = images.to(device)
labels = labels.to(device)

print("Image shape:", tuple(images.shape))
print("Image dtype:", images.dtype)
print("Image range:", images.min().item(), "to", images.max().item())
print("Image mean:", images.mean().item())
print("Image std:", images.std().item())
print("Label range:", labels.min().item(), "to", labels.max().item())

with torch.no_grad():
    logits = baseline_model(images)
    preds = logits.argmax(dim=1)
    print("Baseline accuracy on this batch:",
          f"{100 * (preds == labels).float().mean().item():.2f}%")

    # Test common input scaling variants on the same images.
    variants = {
        "current": images,
        "scaled_minus1_to1": images * 2 - 1,
    }

    for name, x in variants.items():
        out = baseline_model(x)
        acc = (out.argmax(dim=1) == labels).float().mean().item()
        print(f"{name} batch accuracy: {100 * acc:.2f}%")

print("\nDiagnostic only: no training and no checkpoint changes.")


Image shape: (32, 1, 224, 224)
Image dtype: torch.float32
Image range: 0.0 to 1.0
Image mean: 0.35118287801742554
Image std: 0.13116547465324402
Label range: 0 to 15
Baseline accuracy on this batch: 0.00%
current batch accuracy: 0.00%
scaled_minus1_to1 batch accuracy: 0.00%

Diagnostic only: no training and no checkpoint changes.


In [17]:

# NOTEBOOK 10 — CELL 11: CHECK BASELINE WITH NORMALIZED INPUTS
import torch

assert "baseline_model" in globals()
assert "val_loader" in globals()

baseline_model.eval()
device = next(baseline_model.parameters()).device

correct, total = 0, 0

with torch.no_grad():
    for step, (images, labels) in enumerate(val_loader, start=1):
        images = images.to(device)
        labels = labels.to(device)

        # Apply only if the loader currently supplies pixels in [0, 1].
        if images.min().item() >= 0 and images.max().item() <= 1:
            images = (images - 0.5) / 0.5

        predictions = baseline_model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        if step % 25 == 0 or step == len(val_loader):
            print(f"Batch {step}/{len(val_loader)}")

print(f"\nBaseline validation accuracy: {100 * correct / total:.2f}%")
print("Reference from saved history: 42.175%")
print("No training performed; no checkpoints modified.")


Batch 25/125
Batch 50/125
Batch 75/125
Batch 100/125
Batch 125/125

Baseline validation accuracy: 0.05%
Reference from saved history: 42.175%
No training performed; no checkpoints modified.


In [19]:

# NOTEBOOK 10 — CELL 12 FIX: ORIGINAL BASELINE PREPROCESSING

import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
SPLIT_CSV = PROJECT / "results/CASIA_Thousand_data_split.csv"
CHECKPOINT_PATH = PROJECT / "models/CASIA_Thousand_baseline_resnet18.pth"

split = pd.read_csv(SPLIT_CSV)
val_rows = split[split["split"] == "validation"].copy().reset_index(drop=True)

# Find the image-path column in the original split file
path_col = next(
    (c for c in val_rows.columns
     if c.lower() in ["image_path", "path", "filepath", "file_path"]),
    None
)
if path_col is None:
    raise ValueError(f"No image path column found. Columns: {split.columns.tolist()}")

# Original Notebook 04 preprocessing
original_transform = transforms.Compose([
    transforms.CenterCrop(480),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

class OriginalBaselineValidationDataset(Dataset):
    def __init__(self, dataframe, transform, path_column):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.path_column = path_column

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = Path(str(row[self.path_column]))

        # Resolve paths if the CSV contains paths from another environment.
        if not image_path.is_file():
            matches = list(Path("/content/ocular_project_local").rglob(image_path.name))
            if matches:
                image_path = matches[0]
            else:
                raise FileNotFoundError(f"Image not found: {row[self.path_column]}")

        with Image.open(image_path) as im:
            image = im.convert("L")
            image = self.transform(image)

        # Use the exact class mapping currently used in Notebook 10.
        identity = str(row["identity"])
        label = label_to_idx[identity]
        return image, label

original_val_loader = DataLoader(
    OriginalBaselineValidationDataset(val_rows, original_transform, path_col),
    batch_size=32, shuffle=False, num_workers=2
)

baseline_original = models.resnet18(weights=None)
baseline_original.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
baseline_original.fc = nn.Linear(
    baseline_original.fc.in_features, NUM_CLASSES
)

state = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]
elif isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

baseline_original.load_state_dict(state, strict=True)
baseline_original = baseline_original.to(DEVICE).eval()

correct, total = 0, 0
with torch.no_grad():
    for step, (images, labels) in enumerate(original_val_loader, 1):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        predictions = baseline_original(images).argmax(1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        if step % 25 == 0 or step == len(original_val_loader):
            print(f"Batch {step}/{len(original_val_loader)}")

print(f"\nOriginal-pipeline validation accuracy: {100 * correct / total:.2f}%")
print("Reference from baseline history: 42.175%")
print("No training performed; checkpoints unchanged.")


Batch 25/125
Batch 50/125
Batch 75/125
Batch 100/125
Batch 125/125

Original-pipeline validation accuracy: 42.12%
Reference from baseline history: 42.175%
No training performed; checkpoints unchanged.


In [20]:

# NOTEBOOK 10 — FINAL BASELINE TEST VERIFICATION
import json
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
SPLIT_CSV = PROJECT / "results/CASIA_Thousand_data_split.csv"
CHECKPOINT = PROJECT / "models/CASIA_Thousand_baseline_resnet18.pth"
OUT_DIR = PROJECT / "results/recognition_with_augmented_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Original Notebook 04 test split and preprocessing
split = pd.read_csv(SPLIT_CSV)
test_df = split[split["split"] == "test"].copy().reset_index(drop=True)

path_col = next(
    c for c in test_df.columns
    if c.lower() in ["image_path", "path", "filepath", "file_path"]
)

original_transform = transforms.Compose([
    transforms.CenterCrop(480),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

class BaselineTestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = Path(str(row[path_col]))

        if not p.is_file():
            matches = list(
                Path("/content/ocular_project_local").rglob(p.name)
            )
            if not matches:
                raise FileNotFoundError(f"Test image not found: {p}")
            p = matches[0]

        with Image.open(p) as im:
            image = original_transform(im.convert("L"))

        identity = str(row["identity"])
        label = label_to_idx[identity]
        return image, label

test_loader_original = DataLoader(
    BaselineTestDataset(test_df),
    batch_size=32,
    shuffle=False,
    num_workers=2
)

model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

state = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
if "model_state_dict" in state:
    state = state["model_state_dict"]
elif "state_dict" in state:
    state = state["state_dict"]

model.load_state_dict(state, strict=True)
model = model.to(DEVICE).eval()

correct, total = 0, 0
with torch.no_grad():
    for step, (images, labels) in enumerate(test_loader_original, 1):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.numel()

        if step % 25 == 0 or step == len(test_loader_original):
            print(f"Batch {step}/{len(test_loader_original)}")

accuracy = correct / total
result = {
    "experiment": "Original baseline, original preprocessing",
    "test_samples": total,
    "correct_predictions": correct,
    "test_accuracy": accuracy,
    "test_accuracy_percent": round(100 * accuracy, 4),
    "preprocessing": "CenterCrop(480), Resize(256,256), ToTensor; no normalization"
}

save_path = OUT_DIR / "notebook10_baseline_original_pipeline_test.json"
with open(save_path, "w") as f:
    json.dump(result, f, indent=2)

print("\n===== BASELINE TEST VERIFICATION =====")
print(f"Test samples: {total}")
print(f"Correct predictions: {correct}")
print(f"Baseline test accuracy: {100 * accuracy:.2f}%")
print("Historical baseline test accuracy: 30.18%")
print(f"Saved: {save_path}")
print("No training performed; checkpoint unchanged.")


Batch 25/125
Batch 50/125
Batch 75/125
Batch 100/125
Batch 125/125

===== BASELINE TEST VERIFICATION =====
Test samples: 4000
Correct predictions: 1207
Baseline test accuracy: 30.18%
Historical baseline test accuracy: 30.18%
Saved: /content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/notebook10_baseline_original_pipeline_test.json
No training performed; checkpoint unchanged.


In [21]:

# NOTEBOOK 10 — FINAL COMPARISON, EXPORT & AUDIT

import json
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Ocular_Project")
RESULTS = PROJECT / "results" / "recognition_with_augmented_data"
RESULTS.mkdir(parents=True, exist_ok=True)

# Previously verified results
baseline = {
    "experiment": "Original baseline",
    "test_samples": 4000,
    "accuracy": 0.30175,
    "macro_precision": 0.272411,
    "macro_recall": 0.30175,
    "macro_f1": 0.260661,
    "preprocessing": "CenterCrop(480), Resize(256,256), ToTensor; no normalization"
}

augmented = {
    "experiment": "Diffusion-augmented recognition",
    "test_samples": 4000,
    "accuracy": 0.55825,
    "macro_precision": 0.504488,
    "macro_recall": 0.55825,
    "macro_f1": 0.503830,
    "preprocessing": "Notebook 10 evaluation pipeline",
}

# Confirm that the evaluation artifacts exist
required_files = [
    RESULTS / "notebook10_baseline_original_pipeline_test.json",
    RESULTS / "notebook10_test_metrics.json",
    RESULTS / "notebook10_test_predictions.csv",
    RESULTS / "notebook10_per_class_metrics.csv",
    RESULTS / "notebook10_confusion_matrix.npy",
    RESULTS / "notebook10_training_history.csv",
]

print("Artifact existence audit:")
for path in required_files:
    print(f"{'OK' if path.exists() else 'MISSING'}: {path.name}")

comparison = pd.DataFrame([baseline, augmented])
comparison["accuracy_percent"] = comparison["accuracy"] * 100
comparison["macro_precision_percent"] = comparison["macro_precision"] * 100
comparison["macro_recall_percent"] = comparison["macro_recall"] * 100
comparison["macro_f1_percent"] = comparison["macro_f1"] * 100

accuracy_gain_pp = (
    augmented["accuracy"] - baseline["accuracy"]
) * 100

relative_accuracy_gain = (
    (augmented["accuracy"] - baseline["accuracy"])
    / baseline["accuracy"]
) * 100

comparison_path = RESULTS / "notebook10_final_model_comparison.csv"
json_path = RESULTS / "notebook10_final_model_comparison.json"

comparison.to_csv(comparison_path, index=False)

summary = {
    "notebook": "10_Recognition_with_Augmented_Data",
    "test_samples_per_model": 4000,
    "baseline_accuracy_percent": round(baseline["accuracy"] * 100, 2),
    "augmented_accuracy_percent": round(augmented["accuracy"] * 100, 2),
    "accuracy_improvement_percentage_points": round(accuracy_gain_pp, 2),
    "relative_accuracy_improvement_percent": round(relative_accuracy_gain, 2),
    "baseline_preprocessing": baseline["preprocessing"],
    "augmented_preprocessing": augmented["preprocessing"],
    "comparison_caveat": (
        "The baseline was evaluated using its original preprocessing. "
        "The augmented model used the Notebook 10 pipeline. "
        "Therefore, the observed difference is promising but is not a "
        "strictly controlled estimate of the effect of augmentation alone."
    ),
    "retraining_performed": False,
    "comparison_csv": str(comparison_path),
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n===== FINAL COMPARISON =====")
display(comparison[[
    "experiment", "test_samples", "accuracy_percent",
    "macro_precision_percent", "macro_recall_percent",
    "macro_f1_percent"
]].round(2))

print(f"\nAbsolute accuracy improvement: {accuracy_gain_pp:.2f} percentage points")
print(f"Relative accuracy improvement: {relative_accuracy_gain:.2f}%")
print(f"\nSaved CSV:  {comparison_path}")
print(f"Saved JSON: {json_path}")
print("\nComparison artifacts saved. No model was retrained.")


Artifact existence audit:
OK: notebook10_baseline_original_pipeline_test.json
OK: notebook10_test_metrics.json
OK: notebook10_test_predictions.csv
OK: notebook10_per_class_metrics.csv
OK: notebook10_confusion_matrix.npy
OK: notebook10_training_history.csv

===== FINAL COMPARISON =====


,experiment,test_samples,accuracy_percent,macro_precision_percent,macro_recall_percent,macro_f1_percent
0,Original baseline,4000,30.18,27.24,30.18,26.07
1,Diffusion-augmented recognition,4000,55.82,50.45,55.82,50.38



Absolute accuracy improvement: 25.65 percentage points
Relative accuracy improvement: 85.00%

Saved CSV:  /content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/notebook10_final_model_comparison.csv
Saved JSON: /content/drive/MyDrive/Ocular_Project/results/recognition_with_augmented_data/notebook10_final_model_comparison.json

Comparison artifacts saved. No model was retrained.



# Notebook 10: Recognition with Diffusion-Augmented Data

## Objective
The objective of this experiment was to evaluate ocular recognition using a training dataset augmented with synthetic images generated by the identity-conditioned diffusion pipeline.

## Dataset
- Original training images: 12,000
- Synthetic training images: 4,864
- Total augmented training images: 16,864
- Validation images: 4,000
- Test images: 4,000
- Number of identities: 2,000

The validation and test splits were kept unchanged during augmentation.

## Experimental Results

| Metric | Original Baseline | Diffusion-Augmented Model |
|---|---:|---:|
| Test accuracy | 30.18% | 55.83% |
| Macro precision | 27.24%* | 50.45% |
| Macro recall | 30.18%* | 55.83% |
| Macro F1-score | 26.07%* | 50.38% |

*Baseline precision, recall, and F1-score are from the previously saved baseline evaluation.

The augmented model's observed test accuracy was 25.65 percentage points higher than the baseline result.

## Interpretation
The results suggest that training with the augmented dataset may improve recognition performance. However, the baseline was evaluated using its original 256 × 256 preprocessing pipeline, while the augmented model used the Notebook 10 pipeline. Therefore, the difference cannot be attributed exclusively to synthetic augmentation without a controlled comparison using matched preprocessing and verified label mappings.

## Limitations
- Synthetic images may contain visual artifacts or distribution differences from real ocular images.
- Higher recognition accuracy alone does not establish synthetic-image realism or identity preservation.
- The experiment does not by itself establish privacy protection or resistance to membership inference attacks.
- Further controlled experiments and the privacy evaluation are required.

## Conclusion
Notebook 10 completed the training and evaluation of the diffusion-augmented recognition model. The model achieved 55.83% test accuracy, compared with the reproduced baseline accuracy of 30.18%. These results are encouraging, but a matched-preprocessing comparison is required to isolate the effect of augmentation. The findings will be considered alongside the GAN comparison, image-quality analysis, and privacy experiments in the subsequent notebooks.
